# 🧬 00 · Welcome: what machine learning does for biology
### From one gene at a time to genome-scale tables — a guided tour of 02-620 with real leukaemia data, where you only have to press Shift + Enter

**BiologyToML** · Part A — Foundations: from biology to data · Module 00 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/00_welcome_ml_for_biology.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib (no installation needed on Colab) |
| **Prerequisites** | None. High-school algebra and introductory biology. You will only *run* code here; Module 01 teaches you to write it. |
| **Estimated time** | 75 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `001_intro` slides 12–60 (machine learning, the three breakthroughs, supervised vs unsupervised learning, course schedule) — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The leukaemia gene-expression data are **real** (Golub *et al.* 1999, 72 patients × 3,571 genes, preprocessed as
>     in Dudoit *et al.* 2002); if you are offline the notebook falls back to a clearly labelled *simulated* stand-in and
>     says so. The gene-structure picture in Section 2 uses the DNA letters printed on the slides with *illustrative*
>     (made-up) exon/intron labels. The tiny two-gene classifier of Section 4 is a teaching toy, not a diagnostic tool.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Run** a Jupyter/Colab notebook confidently: execute cells in order, restart, and recover from mistakes.
2. **Describe** the three breakthroughs of `001_intro` (genome sequence, SNP maps, expression profiling) and the machine-learning question each one raised.
3. **Explain** why biological data are stored as a table with **N rows = samples** and **J columns = features**, and read the shape of a real data set.
4. **Distinguish** supervised learning (learn f: X → Y, or P(Y|X)) from unsupervised learning (learn P(X)), and regression from classification.
5. **Interpret** a predicted probability such as P(AML | expression) = 0.9 vs 0.5 as a statement of certainty.
6. **Identify** X, Y and f for gene finding, GWAS, cancer subtyping, gene networks and population structure.
7. **Connect** every lecture of 02-620 to the module of this course that teaches it, and name the three ideas that recur in all of them.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, load the helpers | 5 min |
| 1 · How to use this course | notebooks, cells, the emoji legend, the 25-module roadmap | 12 min |
| 2 · Biology became a data science | the three breakthroughs of slides 18–37 and how big the data are | 12 min |
| 3 · Data as tables | the Golub leukaemia data: 72 patients × 3,571 genes; a heatmap like slide 35 | 10 min |
| 4 · Supervised learning | a two-gene classifier that outputs probabilities (slides 39–48) | 12 min |
| 5 · Unsupervised learning | structure without labels: a 2-D map of all patients (slides 52–57) | 7 min |
| 6 · What is X? What is Y? What is f? | the exercise of slides 49–57 | 6 min |
| 7 · The course map | methods × tasks, the three recurring ideas, the 02-620 schedule | 6 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

In 1999 a patient arrives at a hospital with acute leukaemia: the bone marrow is flooded with immature white blood
cells. Two diseases look alike at this stage. **Acute lymphoblastic leukaemia (ALL)** grows from precursors of
lymphocytes; **acute myeloid leukaemia (AML)** grows from precursors of the myeloid line (granulocytes, monocytes).
They need **different treatments**, so telling them apart matters. Yet no single test settled the question: the
diagnosis combined an experienced haematopathologist's reading of cell shape, staining, surface markers and
chromosomes, each done in a different specialised laboratory.

Todd Golub, Donna Slonim, Eric Lander and colleagues asked a new kind of question. A **microarray** can measure the
activity (expression) of *thousands* of genes in one sample at once. Could a computer **learn** the difference between
ALL and AML from such measurements on patients whose diagnosis was known, and then **predict** the class of new
patients? And could it **discover** the two classes on its own, without being told they exist? Their paper,
*"Molecular classification of cancer: class discovery and class prediction by gene expression monitoring"* (Science,
1999), became one of the founding papers of machine learning in biology — it is on slide 35 of the first 02-620 lecture.

This notebook is your guided tour. You will load the **real Golub data**, see the pattern the authors saw, build a
tiny classifier that outputs probabilities, and watch the two leukaemias separate without labels. Along the way you
will learn how this course works and how its 25 modules map onto the 15 lecture decks of 02-620.

> ▶️ **Never used a notebook before?** A notebook is a document made of *cells*. Grey boxes are **code cells**: click
> inside one and press **Shift + Enter** (or the ▶ button on its left) to run it; its output appears underneath. White
> boxes like this one are **text cells**. Run the cells **from top to bottom** — each one may use results of the cells
> above it. On Google Colab, click the *Open in Colab* badge at the top, then *Runtime → Run all* if you just want to see
> everything once. Nothing you do here can break your computer, and every mistake can be undone by re-running.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`load_golub` reads the real leukaemia data from the repository (or downloads it, or — offline —
               simulates a stand-in); `standardize` puts genes on a common scale; `accuracy` and
               `confusion_counts` score predictions; `plot_decision_regions` draws what a classifier decides.
               You do not need to read this code today. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_DATA_URL = "https://raw.githubusercontent.com/xulabs/edu/main/ml/tutorial/BiologyToML/datasets/{name}"


CACHE_DIR = os.environ.get("BIO2ML_CACHE", "course_data")


def fetch_course_file(name: str, timeout: float = 60.0) -> str | None:
    """Return a local path to the course data file ``name``, or ``None`` if it cannot be found.

    Looks in the repository's ``datasets/`` folder first (when the notebook runs inside a
    clone of xulabs/edu), then in the download cache, and finally downloads it from GitHub.
    """
    for base in ("../datasets", "datasets", "ml/tutorial/BiologyToML/datasets", CACHE_DIR):
        path = os.path.join(base, name)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return path
    dest = os.path.join(CACHE_DIR, name)
    os.makedirs(CACHE_DIR, exist_ok=True)
    try:
        with urllib.request.urlopen(REPO_DATA_URL.format(name=name), timeout=timeout) as r, \
                open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return dest
    except Exception as e:  # noqa: BLE001 — any network problem → offline fallback
        print(f"⚠️  Could not obtain {name} ({type(e).__name__}); using a simulated stand-in instead.")
        return None


def simulate_golub_like(rng: np.random.Generator) -> dict:
    """Simulated stand-in for the Golub data: 72 samples (47 'ALL', 25 'AML') × 3,571 genes.

    Log10-scale expression with gene-specific means, correlated gene modules and
    60 genes that differ between the classes.  Same keys as ``load_golub``.
    """
    n_all, n_aml, J = 47, 25, 3571
    y = np.r_[np.zeros(n_all, int), np.ones(n_aml, int)]
    base = rng.normal(2.6, 0.45, J)
    modules = rng.normal(0, 1, (len(y), 20))                  # 20 hidden "programmes"
    loading = rng.normal(0, 0.15, (20, J)) * (rng.random((20, J)) < 0.1)
    X = base + modules @ loading + rng.normal(0, 0.25, (len(y), J))
    informative = rng.choice(J, 60, replace=False)
    X[:, informative] += np.outer(y - 0.5, rng.choice([-1, 1], 60) * rng.uniform(0.4, 0.9, 60))
    is_train = np.r_[np.arange(n_all) < 27, np.arange(n_aml) < 11]
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL"), genes=np.array([f"gene{j:04d}" for j in range(J)]),
                is_train=is_train, sample_id=np.arange(1, len(y) + 1), source="SIMULATED stand-in (offline)")


def load_golub() -> dict:
    """Golub et al. (1999) leukaemia expression data, preprocessed as in Dudoit et al. (2002).

    Returns a dict with ``X`` (72 × 3,571, log10 expression), ``y`` (0 = ALL, 1 = AML),
    ``labels``, ``genes`` (Affymetrix probe IDs), ``is_train`` (the original 38/34 split),
    ``sample_id`` and ``source``.  Falls back to ``simulate_golub_like`` when offline.

    Raw intensities were clipped to [100, 16000] before the log10, so the value 2.0 means
    "at or below the detection limit" (about 20 % of all entries).  Some genes are therefore
    constant within a class — per-class variance estimates need a small floor.
    """
    path = fetch_course_file("golub_leukemia.csv.gz")
    if path is None:
        return simulate_golub_like(np.random.default_rng(1999))
    df = pd.read_csv(path)
    return dict(X=df.iloc[:, 3:].to_numpy(float), y=(df["label"] == "AML").to_numpy(int),
                labels=df["label"].to_numpy(), genes=np.array(df.columns[3:]),
                is_train=(df["golub_split"] == "train").to_numpy(), sample_id=df["sample_id"].to_numpy(),
                source="Golub et al. 1999 (real data)")


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def plot_decision_regions(predict, X, y, ax=None, resolution: int = 200, title: str = "",
                          class_names=None, pad: float = 0.5):
    """Shade the regions a 2-feature classifier assigns to each class and overlay the data.

    ``predict`` maps an (M, 2) array to M integer labels 0, 1, 2, …
    """
    X, y = np.asarray(X, float), np.asarray(y).astype(int)
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - pad, X[:, 0].max() + pad
    y0, y1 = X[:, 1].min() - pad, X[:, 1].max() + pad
    gx, gy = np.meshgrid(np.linspace(x0, x1, resolution), np.linspace(y0, y1, resolution))
    Z = np.asarray(predict(np.c_[gx.ravel(), gy.ravel()])).astype(int).reshape(gx.shape)
    k = max(int(y.max()), int(Z.max())) + 1
    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(CLASS_COLORS[:k])
    ax.contourf(gx, gy, Z, levels=np.arange(k + 1) - 0.5, cmap=cmap, alpha=0.18)
    ax.contour(gx, gy, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=0.8)
    for c in range(k):
        m = y == c
        if m.any():
            ax.scatter(X[m, 0], X[m, 1], s=18, color=CLASS_COLORS[c], edgecolor="k", linewidth=0.3,
                       label=(class_names[c] if class_names is not None else f"class {c}"))
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1)
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['fetch_course_file', 'simulate_golub_like', 'load_golub', 'standardize', 'accuracy', 'confusion_counts', 'plot_decision_regions']))

---
## 1 · How to use this course

> 📍 **Where we are.** This is the very first module. Before any biology or mathematics, let us make sure the tool —
> the notebook — feels comfortable, and see where the 25 modules are going.

### 1.1 Cells, the kernel and the order of execution

Behind every notebook runs a Python program called the **kernel**. When you run a code cell, the kernel executes it
and *remembers* everything the cell created (numbers, tables, functions). The number in square brackets to the left
of a cell, e.g. `[3]`, tells you the order in which cells were run. Three habits will save you hours:

1. **Run cells in order, from the top.** A cell that uses a variable defined further up will fail if the cell above
   has not been run yet.
2. **If things get strange, restart.** *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All*
   (Jupyter) wipes the kernel's memory and runs everything again from a clean state.
3. **Errors are normal.** A red error message is the computer telling you exactly what went wrong; Module 01 teaches
   you to read them. Fix, re-run, move on.

Try it: run the next cell. It does a little genome arithmetic — the human genome has about 3 billion bases (slide 20)
and roughly 20,000 protein-coding genes.

In [ ]:
# ── 1.1 Your first code cell: genome arithmetic ──
genome_bases = 3_000_000_000          # ≈ 3 billion bases in one copy of the human genome (slide 20)
protein_coding_genes = 20_000         # roughly 20,000 protein-coding genes
print("Hello, biologist!")
print("Bases of genome per protein-coding gene, on average:", genome_bases // protein_coding_genes)

🔍 **What just happened?** The kernel stored two numbers under the names `genome_bases` and `protein_coding_genes`
(such names are called **variables**), divided one by the other and printed the result: about **150,000 bases per
gene**. Genes are scattered sparsely through a sea of non-coding DNA — which is exactly why *finding* them (Section 2)
was a hard problem. Lines starting with `#` are **comments**: notes for humans that the computer ignores.

In [ ]:
# ── 1.2 Cells remember: use the variables defined above ──
n_ALL, n_AML = 47, 25                 # patients of each leukaemia type in the Golub data set
n_patients = n_ALL + n_AML
print(f"{n_patients} patients, of whom {n_AML / n_patients:.0%} have AML")
print(f"Bases in one copy of the genome: {genome_bases:,}  (defined in the cell above — the kernel remembered it)")

🔍 **What just happened?** This cell used `genome_bases`, which was created by the *previous* cell — the kernel
remembers it. The odd-looking `f"…{…}"` is a formatted string: Python replaces each `{…}` by its value (`:.0%` shows a
fraction as a percentage, `:,` adds thousands separators).

⚠️ **Order matters.** If you restart the kernel and run cell 1.2 *before* cell 1.1, Python stops with
`NameError: name 'genome_bases' is not defined`. Not a disaster: run cell 1.1, then 1.2 again.

✏️ **Your turn.** In cell 1.2 change `n_AML` to `30` and re-run it with Shift + Enter. The percentage changes. Now set
it back to `25` and re-run, so that later cells see the real number. (You have just done what scientists do all day:
change one thing, re-run, compare.)

### 1.2 The anatomy of every notebook and the emoji legend

All 25 notebooks share one structure, so you always know where to look:

| Symbol | Meaning |
|---|---|
| 🎯 / 🗺️ | learning objectives and the contents table (with minutes per section) |
| 🧫 | the biological question that motivates the module |
| 📍 **Where we are** | how a section connects to what came before |
| 🧮 **Math refresher** | a gentle (re)introduction to a piece of mathematics, assuming you may never have seen it |
| *In words:* | a plain-language reading that follows **every** equation |
| 🔍 **What just happened?** | interpretation of the output you just produced — compare it with what *you* see |
| ✏️ **Your turn** | a concrete change for you to make, and a question to answer |
| ⚠️ | a pitfall or common misconception |
| 📑 **Slide connection** | which 02-620 slides the section teaches, what the notebook adds, and an exam tip |
| ✅ / 🧠 / 🚀 / 📝 | checkpoint questions with hidden answers, open questions, extensions, and a lab-note template |

**A study workflow that works.** (1) Run the whole notebook once, top to bottom, reading the text. (2) Go back section
by section; re-run each cell and check that the 🔍 text matches your output. (3) Do every ✏️ — *change one parameter
at a time* and predict the result before you run. (4) Fill in the 📝 lab note at the end: writing down what changed
when you modified something is how the material sticks. (5) Open the 02-620 slides named in the 📑 boxes and read them
side by side with the notebook.

### 1.3 The roadmap

The course has five parts. **Part A** (Modules 00–05) teaches the prerequisites of 02-620 — Python, probability,
linear algebra, calculus — *just in time*, always with biological data. Parts B–E follow the 02-620 lectures. Run the
next cell to draw the map.

In [ ]:
# ── 1.3 Draw the 25-module roadmap ──
PARTS_MAP = [
    ("A · Foundations", "tab:blue", ["00 Welcome", "01 Python &\nNumPy", "02 Probability", "03 Linear\nalgebra",
                                     "04 Calculus &\noptimization", "05 Likelihood,\nMLE & MAP"]),
    ("B · Supervised", "tab:orange", ["06 Linear\nregression, GWAS", "07 Ridge &\nlasso", "08 k-NN &\nnaïve Bayes",
                                      "09 Logistic\nregression", "10 Trees &\nforests", "11 Support\nvector machines"]),
    ("C · Unsupervised", "tab:green", ["12 Clustering", "13 Mixtures\n& EM", "14 PCA"]),
    ("D · Graphical models", "tab:purple", ["15 Bayes nets I:\nrepresentation", "16 Bayes nets II:\nindependence",
                                            "17 Bayes nets III:\ninference", "18 Gaussian\ngraphical models",
                                            "19 HMM I:\ndecoding", "20 HMM II:\nlearning"]),
    ("E · Bayes, NN, selection", "tab:red", ["21 Bayesian\nlearning, MCMC", "22 Neural\nnetworks",
                                             "23 Model\nselection", "24 Capstone\nproject"]),
]


def draw_roadmap(parts) -> None:
    """Draw one row of rounded boxes per course part, with arrows from row to row."""
    fig, ax = plt.subplots(figsize=(11.5, 5.2))
    for row, (part, colour, modules) in enumerate(parts):
        y = len(parts) - 1 - row
        ax.text(-0.62, y, part, ha="right", va="center", fontsize=9, fontweight="bold", color=colour)
        for col, name in enumerate(modules):
            ax.add_patch(FancyBboxPatch((col - 0.45, y - 0.36), 0.9, 0.72, boxstyle="round,pad=0.02",
                                        fc=colour, ec="0.3", alpha=0.18 if col else 0.35))
            ax.text(col, y, name, ha="center", va="center", fontsize=7.2)
        if row < len(parts) - 1:            # arrow down to the next part
            ax.annotate("", xy=(0, y - 0.62), xytext=(0, y - 0.38),
                        arrowprops=dict(arrowstyle="-|>", color="0.4"))
    ax.set_xlim(-3.2, 5.6); ax.set_ylim(-0.6, len(parts) - 0.4); ax.axis("off")
    ax.set_title("BiologyToML: 25 modules in five parts (each row builds on the rows above)")
    plt.tight_layout(); plt.show()


draw_roadmap(PARTS_MAP)
print("modules per part:", {p: len(m) for p, _, m in PARTS_MAP}, "→ total", sum(len(m) for _, _, m in PARTS_MAP))

🔍 **What just happened?** Six foundation modules, six supervised-learning modules, three unsupervised, six on
probabilistic graphical models and four on Bayesian learning, neural networks, model selection and the capstone —
**25 in total**. You do not need to understand the code of the cell; it only draws boxes.

**How the modules map onto the 15 lecture decks of 02-620.** Each 02-620 slide deck (the file names below are the deck names
used in every 📑 box) is taught by one or more modules:

| 02-620 deck | Topic | Module(s) |
|---|---|---|
| `001_intro` | machine learning in science; supervised vs unsupervised; MLE vs MAP | **00**, 05 (and 24: the project) |
| `002_regression` | linear regression, GWAS, ridge, lasso | 06, 07 (previews in 03, 04) |
| `003_classification-kNN-NB` | k-nearest neighbours, naïve Bayes | 08 |
| `004_classification-LR` | logistic regression; generative vs discriminative | 09 |
| `005_decision-tree` | decision trees, random forests | 10 |
| `006_clustering` | hierarchical clustering, k-means | 12 |
| `007_SVM` | support vector machines | 11 |
| `008_BN_mz_021825` | Bayesian networks: representation, independence, inference, learning | 15, 16, 17 |
| `009_GGM_mz_030925` | Gaussian graphical models | 18 |
| `010_mixture-model_mz_031725` | mixture models and EM | 13 |
| `011_PCA_mz_031825` | principal component analysis | 14 |
| `012_HMM-DBN_mz_032425` | hidden Markov models, dynamic Bayesian networks | 19, 20 |
| `013_Bayesian-MCMC_mz_032725` | Bayesian learning, MCMC | 21 |
| `014_NN_mz_032725` | neural networks | 22 |
| `015_model-selection_mz_040725` | bias–variance, cross-validation, model selection | 23 (and 07) |

Modules 01–04 cover the two prerequisite courses named on slide 3 (02-601 programming, 02-680 mathematics and
statistics), each tied to the slides where that mathematics is first used.

---
## 2 · Biology became a data science

> 📍 **Where we are.** You can run cells. Now the story of slides 12–37: where machine learning came from, and how three
> technological breakthroughs turned biology into a field that *needs* it. Each breakthrough raised a question that one
> part of this course answers.

**Where machine learning comes from (slides 12–17).** Machine learning grew where **computer science** (algorithms,
data structures) met **statistics** (principles for reasoning from noisy data). The same methods now run speech
recognition, computer vision, translation, self-driving cars, finance — and genomics and medicine. The central idea
is the one babies use to learn what a cat is (slide 14): **learn from examples** instead of being programmed with rules.

**Biology in the old days (slides 18–19).** Until about 2000, a lab typically studied *one gene at a time* — a
candidate gene with some evidence linking it to a disease. Mutations in the **HTT** (huntingtin) gene cause Huntington
disease (the slide's "ATT gene" is almost certainly a typo for HTT); mutations in **CFTR** cause cystic fibrosis, and CFTR was sequenced
in many patients. But what about the other ~20,000 genes? Studying them one by one was impossible.

### Breakthrough 1 — the human genome sequence (2001, slides 20–25)

In 2001 the first draft of the human genome — **about 3 billion nucleotides** — became public, assembled and served
through the UCSC Genome Browser built by Jim Kent and David Haussler. The first question was: *where are the genes?*
Slide 21 shows what a computer actually receives: a string of letters.

In [ ]:
# ── 2.1 The genome, as a computer sees it ──
slide21 = ("cggtgaaactgcacgattgttgctggcttaaagatagaccaatcagagtgtgtaacgtc"
           "tatttagcgtcttctatcatccaatcactgcactttacacactataaatagagcagctcag"
           "tgggcgtatttgcgctagtgttgggtgttccgctgtgctgtttttccgtcatggctcgcatc"
           "ctaagcaaactgctcggaagtctactggtggcaaggcgccacgcaaacagttggccac")   # first lines of slide 21
print(slide21[:60], "...")
print("letters shown:", len(slide21))
for base in "acgt":
    print(f"  {base}: {slide21.count(base):3d}  ({slide21.count(base) / len(slide21):.0%})")
print(f"fraction of the whole genome this represents: {len(slide21) / genome_bases:.1e}")

🔍 **What just happened?** The slide's letters are just a string: the cell counted each base and found that this
short stretch is about **one ten-millionth** of one genome. Nothing in the letters says "gene starts here". Biologists
knew that genes have a structure — **exons** (coding pieces) separated by **introns**, with long **intergenic**
stretches between genes (slide 22) — so the task is to *colour* every letter with its hidden role. Run the next cell
to see what that means.

In [ ]:
# ── 2.2 Gene finding = giving every letter a hidden label (illustrative labels, not the real annotation) ──
segments = [("intergenic", 30), ("exon", 18), ("intron", 25), ("exon", 10), ("intron", 20), ("exon", 15),
            ("intergenic", 35), ("exon", 22), ("intron", 15), ("exon", 14), ("intergenic", 36)]
colours = {"intergenic": "tab:blue", "intron": "gold", "exon": "tab:red"}
fig, ax = plt.subplots(figsize=(11, 2.3))
start = 0
for label, length in segments:
    ax.add_patch(plt.Rectangle((start, 0), length, 0.8, color=colours[label]))
    start += length
for i, base in enumerate(slide21[:start]):            # the observed letters, one per position
    if i % 2 == 0:                                    # every second letter, so that they stay readable
        ax.text(i + 0.5, 1.25, base, ha="center", va="center", fontsize=5.5, family="monospace")
ax.text(-3, 1.25, "observed X\n(letters)", ha="right", va="center", fontsize=8)
ax.text(-3, 0.4, "hidden Y\n(labels)", ha="right", va="center", fontsize=8)
for k, (label, c) in enumerate(colours.items()):
    ax.add_patch(plt.Rectangle((start + 8, 1.4 - 0.5 * k), 6, 0.35, color=c))
    ax.text(start + 16, 1.57 - 0.5 * k, label, va="center", fontsize=8)
ax.set_xlim(-30, start + 40); ax.set_ylim(-0.3, 1.8); ax.axis("off")
ax.set_title(f"Gene finding: {start} observed letters, each with a hidden label (labels are ILLUSTRATIVE)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The top row is what the sequencer gives us — the letters (we print every second one). The
coloured bar is what we *want*: the role of each letter. The labels here are invented for illustration; real gene
finders must infer them. The insight that the lecture (slide 24) credits to David Haussler is that this is the *same problem* as **speech
recognition**: a recording is a long signal in which hidden words must be segmented and labelled ("I love the Santa
Cruz beach"). The algorithms developed for speech — **hidden Markov models**, a kind of probabilistic graphical model —
could be borrowed for genomes. You will build exactly such a gene finder in [Module 19](19_hmm_inference.ipynb).
Lesson of slide 23: *recognise that your biological problem has been solved elsewhere, and borrow the method.*

### Breakthrough 2 — maps of human variation (2001–today, slides 26–32)

Any two people differ at millions of positions. A position where some people carry one base and others another is a
**single-nucleotide polymorphism (SNP)**. In 2001 a SNP map listed **1.42 million** SNPs; in 2005 **HapMap Phase I**
genotyped about **1.3 million** SNPs in **270 people** from four populations (30 Yoruba trios from Ibadan, Nigeria; 30
trios of European ancestry from Utah; 45 Han Chinese from Beijing; 45 Japanese from Tokyo). The **1000 Genomes Project**
(2010) sequenced whole genomes, hospitals began sequencing newborns (BabySeq), and **UK Biobank** follows **500,000**
volunteers with genotypes and health records. Two questions arose:

* *Which variants cause disease?* (slide 30) — comparing genotypes of healthy and affected people is a **regression or
  classification** problem: genome-wide association studies, Modules 06–11.
* *What does a genome say about ancestry?* (slides 31–32) — Novembre *et al.* (2008) analysed genotypes at **197,146**
  SNPs in **1,387** Europeans and compressed each person to **2 numbers** by **principal component analysis**; the resulting
  scatter plot looks like a map of Europe. That is **dimensionality reduction**, Module 14.

### Breakthrough 3 — measuring all genes at once (1999–today, slides 33–37)

DNA tells you what *could* happen; to understand *mechanism* you must see which genes are **expressed**. **Microarrays**
measured the expression of thousands of genes per sample (slide 34) — Golub's leukaemia study (slide 35, **clustering
and classification**, Modules 08–14) and the learning of **gene regulatory networks** (slide 36, **probabilistic
graphical models**, Modules 15–18). Today **RNA-seq** counts the messenger RNAs directly, and **single-cell RNA-seq**
(slide 37) does so in each of tens of thousands of individual cells.

How big are these data? Run the next cell.

In [ ]:
# ── 2.3 How many numbers does each breakthrough produce? (sizes quoted on the 02-620 slides) ──
studies = pd.DataFrame([
    ("Human genome draft (2001)", "1 genome × 3 billion bases", 3e9, "where are the genes? → HMMs", "19–20"),
    ("HapMap Phase I (2005)", "270 people × 1.3 million SNPs", 270 * 1.3e6, "which variants matter? → regression, classification", "06–11"),
    ("Novembre et al. (2008)", "1,387 people × 197,146 SNPs", 1387 * 197_146, "ancestry → PCA", "14"),
    ("Golub et al. (1999)", "72 patients × 7,129 probes", 72 * 7129, "leukaemia subtypes → classification, clustering", "08–14"),
], columns=["study", "shape", "numbers", "question → method", "modules"])
pd.set_option("display.max_colwidth", 60)
print(studies.to_string(index=False, formatters={"numbers": "{:.2e}".format}))
print("\nUK Biobank: 500,000 participants with genotypes and health records (slide 29).")

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.barh(studies["study"], studies["numbers"], color=["tab:blue", "tab:orange", "tab:green", "tab:red"])
ax.set_xscale("log"); ax.set_xlim(1e5, 1e10); ax.invert_yaxis()
ax.set_xlabel("number of measurements in the data set (log scale: each tick = 10× more)")
ax.set_title("Genome-scale biology produces far more numbers than anyone can read")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The data sets range from about **5 × 10⁵** numbers (Golub: 72 × 7,129 = 513,288) to
**3 × 10⁹** letters (one genome); HapMap and the European study each hold a few hundred million genotypes. On a
**logarithmic axis** each tick is ten times the previous one — the only way to fit such different sizes in one picture.
No human can read 300 million genotypes; an algorithm can.

⚠️ **Big is not the same as "many samples".** Look at the *shape*: Golub has only **72 patients** but **7,129
measurements per patient**; Novembre has 1,387 people but 197,146 SNPs each. Biological data are usually **wide**: far
more features than samples. This single fact causes many of the difficulties in this course — least squares breaking
down (Module 07), choosing which genes to trust (Modules 07, 23), and the danger of finding patterns by chance.

### 📑 Slide connection — `001_intro`, slides 12–37

- **What the slides show.** Machine learning sits where computer science and statistics meet (12–13) and learns from examples like a
    child learning "cat" (14); it powers vision, speech, text and robotics (15–17). Biology moved from one gene at a time
    (HTT, CFTR; 18–19) through three breakthroughs: the 2001 genome draft → *where are the genes?* → hidden Markov models
    borrowed from speech recognition (20–25); SNP maps, HapMap (1.3 M SNPs, 270 people), 1000 Genomes, UK Biobank
    (500,000 people) → disease genes by regression/classification and ancestry by PCA (197,146 loci in 1,387 Europeans;
    26–32); microarrays and RNA-seq → Golub's ALL/AML classification, gene networks, single-cell data (33–37).
- **What this notebook adds.** The actual sizes of these data sets side by side on a log scale, the observation that they are *wide*
    (J ≫ N), the letters of slide 21 as a Python string, and a picture of what "labelling every letter" means for gene
    finding.
- **Exam tip.** For each breakthrough be able to name the biological question *and* the family of ML methods the slides
    attach to it (HMM/graphical models; regression/classification and PCA; clustering/classification and graphical
    models). Note the slide's "ATT gene" means HTT.

---
## 3 · Data as tables

> 📍 **Where we are.** Section 2 showed that biology produces huge collections of numbers. Every method in this course
> starts by arranging them as a **table**. Here we load a real one: the Golub leukaemia data.

Almost every data set in 02-620 is a table (in mathematics: a **matrix**) called **X**:

* each **row** is one **sample** — a patient, an individual, a cell. There are **N** rows;
* each **column** is one **feature** — a gene's expression, a SNP's genotype, a protein's level. There are **J** columns;
* the number in row *i*, column *j* is the measurement of feature *j* in sample *i*.

If we also know an outcome for each sample (ALL or AML, healthy or sick, a blood-pressure value), we store it in a
separate column **y** with one entry per row. This **N × J** convention (samples in rows) is the one used on the
02-620 regression slides and throughout this course.

⚠️ **Biologists often draw it the other way round.** The heatmap on slide 35 has *genes as rows* and *patients as
columns*, as most expression papers do. Same data, transposed. Always check which way a table is oriented before you
compute anything.

In [ ]:
# ── 3.1 Load the Golub leukaemia data ──
golub = load_golub()                       # a dictionary: X, y, labels, genes, is_train, sample_id, source
X, y, genes = golub["X"], golub["y"], golub["genes"]
print("Source:", golub["source"])
print("X has shape", X.shape, "→ N =", X.shape[0], "patients (rows), J =", X.shape[1], "genes (columns)")
names_, counts_ = np.unique(golub["labels"], return_counts=True)
print("Classes:", {str(k): int(v) for k, v in zip(names_, counts_)}, "(y = 0 for ALL, 1 for AML)")
print("Original split:", golub["is_train"].sum(), "training and", (~golub["is_train"]).sum(), "test patients")
print(f"Values range from {X.min():.2f} to {X.max():.2f} (log10 of expression, i.e. {10 ** X.min():.0f} to {10 ** X.max():,.0f})")

🔍 **What just happened?** If the first line says *real data*, you are looking at the actual measurements of
**72 patients (47 ALL, 25 AML) × 3,571 genes**. The arrays originally measured 7,129 probes; following Dudoit,
Fridlyand & Speed (2002), values were clipped to the range 100–16,000, genes that barely vary were removed (leaving
3,571), and the numbers were converted to **log10** — so 2.0 means 100 units and 4.2 means about 16,000. The logarithm
turns "twice as much" into "+0.3" wherever you are on the scale. Golub *et al.* used **38 patients to train** and kept
**34 to test**; we keep that split.

Let us peek at a few numbers. The heatmap on slide 35 names its genes; we look up four of them.

In [ ]:
# ── 3.2 A peek at the table: 6 patients × 4 named genes ──
# The 50 genes shown in slide 35 (Golub et al. 1999, Fig. 3B): name → GenBank accession of the probe.
SLIDE35_GENES = {
    "C-myb": "U22376", "Proteasome iota": "X59417", "MB-1": "U05259", "Cyclin D3": "M92287",
    "Myosin light chain": "M31211", "RbAp48": "X74262", "SNF2 (D26156)": "D26156", "HkrT-1": "S50223",
    "E2A": "M31523", "Inducible protein": "L47738", "Dynein light chain": "U32944", "Topoisomerase II β": "Z15115",
    "IRF2": "X15949", "TFIIEβ": "X63469", "Acyl-CoA dehydrogenase": "M91432", "SNF2 (U29175)": "U29175",
    "(Ca2+)-ATPase": "Z69881", "SRP9": "U20998", "MCM3": "D38073", "Deoxyhypusine synthase": "U26266",
    "Op 18": "M31303", "Rabaptin-5": "Y08612", "Heterochromatin p25": "U35451", "IL-7 receptor": "M29696",
    "Adenosine deaminase": "M13792", "Fumarylacetoacetate": "M55150", "Zyxin": "X95735", "LTC4 synthase": "U50136",
    "LYN": "M16038", "HoxA9": "U82759", "CD33": "M23197", "Adipsin": "M84526", "Leptin receptor": "Y12670",
    "Cystatin C": "M27891", "Proteoglycan 1": "X17042", "IL-8 precursor": "Y00787", "Azurocidin": "M96326",
    "p62": "U46751", "CyP3": "M80254", "MCL1": "L08246", "ATPase": "M62762", "IL-8": "M28130",
    "Cathepsin D": "M63138", "Lectin": "M57710", "MAD-3": "M69043", "CD11c": "M81695", "Ebp72": "X85116",
    "Lysozyme": "M19045", "Properdin": "M83652", "Catalase": "X04085",
}
accession = np.array([g.split("_")[0] for g in genes])          # probe 'X95735_at' → accession 'X95735'
NAME_OF = {acc: name for name, acc in SLIDE35_GENES.items()}
col = {name: int(np.flatnonzero(accession == acc)[0]) for name, acc in SLIDE35_GENES.items() if acc in accession}
print(f"{len(col)} of the {len(SLIDE35_GENES)} genes named on slide 35 are among our {X.shape[1]:,} genes")
if len(col) < len(SLIDE35_GENES):            # only with the offline SIMULATED stand-in, which has no real gene names
    print("⚠️ stand-in data: each named gene is replaced by one of the most class-distinguishing columns")
    diff = X[y == 0].mean(axis=0) - X[y == 1].mean(axis=0)
    all_like, aml_like = list(np.argsort(diff)[::-1]), list(np.argsort(diff))
    col = {name: int(all_like.pop(0) if k < 25 else aml_like.pop(0)) for k, name in enumerate(SLIDE35_GENES)}

show = ["Zyxin", "CD33", "C-myb", "MB-1"]
rows = np.r_[0:3, np.flatnonzero(y == 1)[:3]]                   # three ALL and three AML patients
peek = pd.DataFrame(X[np.ix_(rows, [col[g] for g in show])], columns=show,
                    index=[f"patient {golub['sample_id'][r]} ({golub['labels'][r]})" for r in rows])
print(peek.round(2))

🔍 **What just happened?** All 50 genes of the slide's figure survived the filtering, so we can find them by their
accession numbers. In this little 6 × 4 slice, **Zyxin** is clearly higher (larger log10 values) in all three AML
patients (3.35–3.79 vs 2.47–2.49) and **MB-1** is higher in all three ALL patients. **CD33** and **C-myb** point the
same way *on average* but overlap: AML patient 28 has almost the same CD33 value as ALL patient 3, and AML patient 29
has more C-myb than ALL patient 2. The directions make biological sense: CD33 is a surface marker of myeloid cells,
MB-1 (also called CD79A) is part of the B-cell receptor, and C-myb is a transcription factor active in immature blood
cells. But one gene and a handful of patients cannot settle anything; we need to look at *all* patients and *many*
genes.

### 3.3 Which genes differ most between ALL and AML?

Golub *et al.* ranked every gene by a **signal-to-noise score** computed on the training patients only:

$$P(g) = \frac{\mu_{\text{ALL}}(g) - \mu_{\text{AML}}(g)}{\sigma_{\text{ALL}}(g) + \sigma_{\text{AML}}(g)}$$

*In words:* for gene $g$, take the difference between its average expression in ALL and in AML patients ($\mu$, the
Greek letter "mu", stands for an average), and divide by how much it varies within each class ($\sigma$, "sigma", is
the standard deviation — the typical spread). A large positive score means "reliably higher in ALL", a large negative
score "reliably higher in AML". Dividing by the spread rewards genes whose difference is large *compared with the
patient-to-patient variation*. (You will meet means and standard deviations properly in Module 02.)

We compute the score for all 3,571 genes and keep the 25 most ALL-like and the 25 most AML-like, as the slide does.

In [ ]:
# ── 3.3 Rank all genes by Golub's signal-to-noise score (training patients only) ──
def signal_to_noise(values: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Golub's score (mean_ALL − mean_AML) / (sd_ALL + sd_AML) for every column; 0 where both sds are 0."""
    a, b = values[labels == 0], values[labels == 1]
    spread = a.std(axis=0, ddof=1) + b.std(axis=0, ddof=1)
    return np.where(spread > 0, (a.mean(axis=0) - b.mean(axis=0)) / np.where(spread > 0, spread, 1.0), 0.0)


train = golub["is_train"]
score = signal_to_noise(10 ** X[train], y[train])      # on the original (un-logged) expression scale
order = np.argsort(score)                               # from most AML-like (negative) to most ALL-like
top_ALL, top_AML = order[::-1][:25], order[:25]
chosen = np.r_[top_ALL, top_AML]
on_slide = np.isin(accession[chosen], list(SLIDE35_GENES.values()))
print(f"{on_slide.sum()} of our 50 top genes are among the 50 genes of slide 35 "
      f"({on_slide[:25].sum()} ALL-high, {on_slide[25:].sum()} AML-high)")
print("Most ALL-like:", [NAME_OF.get(a, a) for a in accession[top_ALL[:5]]])
print("Most AML-like:", [NAME_OF.get(a, a) for a in accession[top_AML[:5]]])

🔍 **What just happened?** Our ranking recovers **45 of the 50 genes** in the slide's figure — with the same training
patients and the same score, the computer rediscovers the authors' gene list almost exactly. The differences come from
preprocessing: Golub ranked all 6,817 genes before any filtering, and we start from the 3,571 that survived Dudoit's
filters. Even the *order* almost agrees: our five most ALL-like genes (C-myb, proteasome iota, MB-1, cyclin D3,
RbAp48) and five most AML-like genes (fumarylacetoacetate, zyxin, LTC4 synthase, LYN, CD33) are all among the first six
rows of the two blocks on slide 35 (the slide's figure shows only the 38 training patients).

Now the picture of slide 35 — but for **all 72 patients**, including the 34 test patients the ranking never saw.

In [ ]:
# ── 3.4 Heatmap: 50 genes × 72 patients, patients sorted ALL | AML ──
patients = np.r_[np.flatnonzero(y == 0), np.flatnonzero(y == 1)]          # all ALL first, then all AML
Z_genes = standardize(X[:, chosen])[0]                                     # each gene: mean 0, sd 1 across patients
fig, ax = plt.subplots(figsize=(9.5, 7.2))
im = ax.imshow(Z_genes[patients].T, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5, interpolation="nearest")
ax.axvline((y == 0).sum() - 0.5, color="k", lw=1.5); ax.axhline(24.5, color="k", lw=1.5)
names = [NAME_OF.get(a, golub["genes"][j]) for a, j in zip(accession[chosen], chosen)]
ax.set_yticks(range(50)); ax.set_yticklabels(names, fontsize=5.5)
ax.set_xticks([(y == 0).sum() / 2, (y == 0).sum() + (y == 1).sum() / 2]); ax.set_xticklabels(["ALL (47)", "AML (25)"])
ax.set_xlabel("patients (columns, as on slide 35)"); ax.set_ylabel("genes (rows)")
fig.colorbar(im, ax=ax, fraction=0.025, label="expression (z-score: sd above/below the gene's mean)")
ax.set_title(f"{golub['source']}: top-25 ALL-high (upper) and top-25 AML-high (lower) genes")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Red means "higher than this gene's average", blue "lower". The upper block (ALL-high genes)
is red on the left and blue on the right; the lower block (AML-high genes) is the mirror image — the checkerboard of
slide 35. The pattern is not perfect: some patients have an ALL gene switched off, or an AML gene switched on. No single
gene separates the classes flawlessly, but **many genes together** do. Combining many weak clues into one decision is
precisely what a classifier does.

✏️ **Your turn.** In cell 3.3, replace `10 ** X[train]` by `X[train]` (rank on the log scale instead) and re-run cells
3.3 and 3.4. How many of the slide's genes do you recover now? (We found that ranking on the original, un-logged scale
reproduces the slide best — a reminder that "preprocessing choices" can change which genes look important.) Change it
back afterwards.

---
## 4 · Supervised learning: learning a function from labelled examples

> 📍 **Where we are.** We have a table X (expression) and a label y (ALL/AML) for every patient. Slides 38–48 define
> the first big family of machine-learning problems: learn to predict y from X.

**Supervised learning** (slide 39) means learning to perform the task *"given input X, decide output Y"* — learning a
**function** $f: X \to Y$. The data come as **pairs** (X, Y); Y is the *teacher* that supplies the correct answer
(slide 41). Examples from slide 40: blood-test results → healthy (Y = 0) or heart disease (Y = 1); an image → is there a
cat? For us: a patient's expression profile → ALL (Y = 0) or AML (Y = 1).

The workflow has three stages (slides 42–44): **choose** a form for $f$ (a line? a tree? a network?), **learn** it from
the *training data*, and **evaluate** it on *previously unseen* inputs — like an exam whose answers you have not seen,
graded afterwards against the true labels.

Two kinds of supervised problems (slide 45):

* **Regression** — Y is a **continuous** number: a blood-pressure value, a gene's expression level, the PSA level of a
  prostate-cancer patient (Module 06).
* **Classification** — Y is **discrete** (a category): ALL vs AML, healthy vs diseased, cancer subtype (Modules 08–11).

**Modelling uncertainty (slides 46–48).** Instead of a bare answer, we would like $f$ to say *how sure* it is. 02-620
therefore formulates supervised learning as learning a **conditional probability** $P(Y \mid X)$ — read "the
probability of Y given X". For a cat picture, $P(\text{cat} \mid X) = 0.9$ and $P(\text{dog} \mid X) = 0.1$ means
"fairly sure it is a cat"; $0.5$ and $0.5$ means "no idea". The probabilities of all possible answers add up to 1.
For a doctor this is crucial: a confident prediction can be acted on, an uncertain one should be sent for more tests.

Let us build the simplest classifier we can think of, using just **two genes**: Zyxin (AML-high) and C-myb (ALL-high).

In [ ]:
# ── 4.1 Two genes, training and test patients ──
GENE_A, GENE_B = "Zyxin", "C-myb"                     # ✏️ try other names from SLIDE35_GENES, e.g. "CD33", "E2A"
two = X[:, [col[GENE_A], col[GENE_B]]]
test = ~train
Z_train, mu_train, sd_train = standardize(two[train])          # z-scores using the TRAINING mean and sd only
Z_test = standardize(two[test], mu_train, sd_train)[0]         # the test patients are scaled the same way
print(f"training patients: {train.sum()} ({(y[train] == 0).sum()} ALL, {(y[train] == 1).sum()} AML); "
      f"test patients: {test.sum()} ({(y[test] == 0).sum()} ALL, {(y[test] == 1).sum()} AML)")

fig, ax = plt.subplots(figsize=(4.6, 4))
for c, name in enumerate(["ALL", "AML"]):
    m = y[train] == c
    ax.scatter(Z_train[m, 0], Z_train[m, 1], s=30, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=f"{name} (training)")
ax.set_xlabel(f"{GENE_A} expression (z-score)"); ax.set_ylabel(f"{GENE_B} expression (z-score)")
ax.set_title("38 training patients, 2 genes"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Each dot is one training patient, placed by two numbers. We converted each gene to
**z-scores** (how many standard deviations above or below the training average), so that both axes have the same
scale. The AML patients (red) sit mostly at the **lower right** (high Zyxin, low C-myb) and the ALL patients (blue) at
the **upper left**. The clouds overlap only a little.

**A nearest-centroid classifier.** Compute the average position of the ALL patients and of the AML patients — the two
**centroids** (centres of mass). A new patient is assigned to the class whose centroid is **closer**. To turn distances
into probabilities we use

$$P(\text{AML} \mid x) = \frac{e^{-d_{\text{AML}}^2/2}}{e^{-d_{\text{ALL}}^2/2} + e^{-d_{\text{AML}}^2/2}}$$

*In words:* $d_{\text{AML}}$ is the distance from the patient's point $x$ to the AML centroid (and $d_{\text{ALL}}$ to
the ALL centroid); $e^{-d^2/2}$ is large (close to 1) for a nearby centroid and tiny for a distant one; dividing by the
sum makes the two probabilities add up to 1. Equal distances give 0.5/0.5 — maximal uncertainty. (Why exactly this
formula? It is what a simple probability model of each class implies; Modules 08 and 09 derive formulas like it.
Today, just use it.)

In [ ]:
# ── 4.2 Train the nearest-centroid classifier and test it on the 34 unseen patients ──
def fit_centroids(Z: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """Return a (2 × n_features) array: the average point of class 0 (ALL) and of class 1 (AML)."""
    return np.vstack([Z[labels == 0].mean(axis=0), Z[labels == 1].mean(axis=0)])


def centroid_probabilities(Z: np.ndarray, centroids: np.ndarray) -> np.ndarray:
    """P(class | x) ∝ exp(−distance² / 2) to each centroid; each row sums to 1."""
    d2 = ((Z[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)    # squared distance to each centroid
    w = np.exp(-(d2 - d2.min(axis=1, keepdims=True)) / 2)                # subtract the minimum for numerical safety
    return w / w.sum(axis=1, keepdims=True)


centroids = fit_centroids(Z_train, y[train])
p_test = centroid_probabilities(Z_test, centroids)[:, 1]                # P(AML | x) for each test patient
pred_test = (p_test > 0.5).astype(int)
pred_train = (centroid_probabilities(Z_train, centroids)[:, 1] > 0.5).astype(int)
print("centroids (z-scores):", dict(ALL=centroids[0].round(2), AML=centroids[1].round(2)))
print(f"training accuracy: {accuracy(y[train], pred_train):.3f}")
print(f"test accuracy:     {accuracy(y[test], pred_test):.3f}  "
      f"({int(np.sum(pred_test == y[test]))} of {test.sum()} unseen patients correct)")
print("test confusion counts (positive = AML):", confusion_counts(y[test], pred_test))

🔍 **What just happened?** "Training" here means computing two averages — the whole learned model is four numbers.
Yet on the **34 patients it never saw**, the classifier gets **31 right (91 %)**; on the training patients it scores
0.974. The confusion counts tell us *which* mistakes were made: an AML patient called ALL is a **false negative** (FN)
and an ALL patient called AML a **false positive** (FP), with "positive" meaning AML. Here every AML patient was
recognised (TP = 14, FN = 0) and all three errors are ALL patients predicted to be AML (FP = 3) — in medicine, *which*
error you make matters as much as how many.

It is normal for test accuracy to be lower than training accuracy: the model was fitted to the training patients.
**Only the test score tells us how the classifier would do on new patients** — which is why 02-620 insists on
evaluating on unseen data (slide 44).

In [ ]:
# ── 4.3 What the classifier decides everywhere, and how sure it is ──
def predict_label(points: np.ndarray) -> np.ndarray:
    """Class (0 = ALL, 1 = AML) of the nearer centroid, for an (M × 2) array of z-scored points."""
    return (centroid_probabilities(points, centroids)[:, 1] > 0.5).astype(int)


fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4))
plot_decision_regions(predict_label, Z_train, y[train], ax=a1, class_names=["ALL (train)", "AML (train)"])
for c in (0, 1):
    m = y[test] == c
    a1.scatter(Z_test[m, 0], Z_test[m, 1], marker="s", s=34, facecolor="none", edgecolor=CLASS_COLORS[c], lw=1.4,
               label=["ALL (test)", "AML (test)"][c])
a1.scatter(*centroids.T, marker="X", s=140, color="k", label="centroids")
a1.set_xlabel(f"{GENE_A} (z-score)"); a1.set_ylabel(f"{GENE_B} (z-score)"); a1.legend(fontsize=6.5, loc="lower left")
a1.set_title("decision regions (shaded) learned from the 38 training patients")

order_p = np.argsort(p_test)
a2.axhspan(0.3, 0.7, color="0.9", label="uncertain (0.3–0.7)")
a2.scatter(range(test.sum()), p_test[order_p], c=[CLASS_COLORS[c] for c in y[test][order_p]], edgecolor="k", lw=0.3, s=30)
a2.axhline(0.5, color="k", ls="--", lw=1)
a2.set_xlabel("test patients, sorted by predicted probability"); a2.set_ylabel("P(AML | expression)")
a2.set_title("blue = truly ALL, red = truly AML"); a2.legend(loc="upper left", fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** *Left:* the classifier splits the plane by a straight line, halfway between the two
centroids (black crosses); every point on the red side is called AML. Open squares are the test patients: most fall on
the correct side, but three blue squares (ALL patients) sit on the red side. *Right:* the predicted probability for
each test patient. Most are confidently near 0 (ALL) or 1 (AML), like the "cat 0.9 / dog 0.1" of slide 47; five fall in
the grey **uncertain band**, like the "0.5 / 0.5" of slide 48. Look at the three blue dots above the dashed line: those
are the errors.

In [ ]:
# ── 4.4 The most uncertain test patients ──
results = pd.DataFrame({"patient": golub["sample_id"][test], "true class": golub["labels"][test],
                        "P(AML|x)": p_test.round(3), "prediction": np.where(pred_test == 1, "AML", "ALL")})
results["|P − 0.5|"] = (results["P(AML|x)"] - 0.5).abs().round(3)
print("The six test patients closest to 0.5:")
print(results.sort_values("|P − 0.5|").head(6).to_string(index=False))
print("\nThe misclassified test patients:")
print(results[results["true class"] != results["prediction"]].to_string(index=False))

🔍 **What just happened?** The first table lists the test patients whose probability is closest to 0.5 — the ones a
cautious doctor would send for more tests. One of the three errors (patient 55, P = 0.56) is among them: there the model
*knew* it was unsure. But the second table shows a sobering truth: patients 67 and 71 are ALL, yet the model gave them
P(AML | x) = 0.82 and 0.87 — **confidently wrong**. Their Zyxin and C-myb values simply look AML-like. A probability
expresses uncertainty only *within the model*; a model that sees only two genes cannot know what it is missing. This is
why 02-620 models $P(Y \mid X)$ rather than a bare $f(X)$ — and why the *model* behind the probability must be good.

✏️ **Your turn.** In cell 4.1 set `GENE_A, GENE_B = "CD33", "E2A"` and re-run cells 4.1–4.4. What is the test accuracy
now, and how many patients are uncertain? Then try `"MB-1"` instead of `"E2A"`. Which pair works best — and why would it
be **cheating** to pick the pair by looking at the *test* accuracy? (Module 23 calls this *selection bias*: once you
use the test patients to make choices, they no longer measure performance on new patients.)

⚠️ **Two cautions.** (1) A probability from a toy model is not a calibrated clinical risk: our 0.9 means "closer to the
AML centroid", nothing more. (2) Golub *et al.* used 50 genes and a weighted vote, and reported that 29 of the 34 test
patients were predicted with high confidence, all of them correctly; our two-gene toy only illustrates the idea.

### 📑 Slide connection — `001_intro`, slides 38–48

- **What the slides show.** Supervised learning learns a function f: X → Y from training pairs (X, Y), where Y is the teacher's correct
    answer (39–43); the learned function is then applied to previously unseen inputs and graded against the truth
    (44). Continuous Y = regression, discrete Y = classification (45). To express uncertainty, learning f is
    formulated as learning P(Y|X) (46), e.g. P(cat|X) = 0.9, P(dog|X) = 0.1 (confident) vs 0.5/0.5 (uncertain), with the
    probabilities summing to 1 (47–48).
- **What this notebook adds.** A real ALL/AML classifier trained on the 38 Golub training patients and evaluated on the 34 test patients:
    decision regions, test accuracy, the kinds of errors (FN/FP), patients flagged as uncertain (P ≈ 0.5) — and two
    patients that are *confidently* wrong, a reminder that a probability is only as good as the model behind it. The speaker notes' "two issues" — choose the functional form, then learn it — are visible: our
    form is "nearer centroid", learning = averaging.
- **Exam tip.** Be able to classify a problem as regression or classification from the type of Y alone (not X), and to
    explain why P(Y|X) is more useful than a hard label. Remember: probabilities of all classes given the same X sum
    to 1 — P(cat|X) + P(dog|X) = 1, not P(cat|X) + P(cat|X′).

---
## 5 · Unsupervised learning: structure without labels

> 📍 **Where we are.** Section 4 used the labels. But the title of Golub's paper also promises *class discovery*: could
> we have found the two leukaemias without being told? Slides 52–57 define the second family of problems.

**Unsupervised learning** (slides 52–53) means *"find interesting patterns in X"* — there is **no Y**, no teacher.
Examples: group images of animals, or newspaper articles by topic, without anyone labelling them. Because unlabelled
data are cheap and plentiful, this is enormously useful — and, as slide 52 says, **much harder** than supervised
learning: without correct answers it is not even obvious how to measure success (slide 60).

In probability language (slide 55), supervised learning models $P(Y \mid X)$, while unsupervised learning models the
**distribution of the data themselves**, $P(X)$ — which values of X are common, which are rare, which groups exist.

Below, a method called **principal component analysis (PCA)** squeezes each patient's 3,571 numbers into just **2**
numbers that keep as much of the patient-to-patient variation as possible — exactly what Novembre *et al.* did with
197,146 SNPs (slides 32, 57). Treat PCA as a black box today; [Module 14](14_pca_dimensionality_reduction.ipynb) opens
it. Crucially, **PCA never sees the labels**: we colour the points only *afterwards*.

In [ ]:
# ── 5.1 A 2-D map of all 72 patients from all 3,571 genes — computed WITHOUT the labels ──
def pca_scores(data: np.ndarray, k: int = 2):
    """Black box until Module 14: coordinates of each row on the k main axes of variation, and the share of
    the total variance that each axis captures."""
    centred = data - data.mean(axis=0)
    U, S, Vt = np.linalg.svd(centred, full_matrices=False)
    # an axis has no natural +/− direction: fix its sign (without labels) so every computer draws the same map
    signs = np.sign(Vt[np.arange(k), np.argmax(np.abs(Vt[:k]), axis=1)])
    return U[:, :k] * S[:k] * signs, (S ** 2 / np.sum(S ** 2))[:k]


scores, share = pca_scores(X)                           # note: y is not used here
print(f"axis 1 keeps {share[0]:.1%} and axis 2 keeps {share[1]:.1%} of the variation in all 3,571 genes")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
a1.scatter(scores[:, 0], scores[:, 1], s=26, color="0.5", edgecolor="k", lw=0.3)
a1.set_title("what the algorithm sees: 72 patients, no labels")
for c, name in enumerate(["ALL", "AML"]):
    m = y == c
    a2.scatter(scores[m, 0], scores[m, 1], s=26, color=CLASS_COLORS[c], edgecolor="k", lw=0.3, label=name)
a2.set_title("the same map, coloured AFTERWARDS by diagnosis"); a2.legend()
for a in (a1, a2):
    a.set_xlabel("PC 1 (main axis of variation)"); a.set_ylabel("PC 2")
plt.tight_layout(); plt.show()
print("average PC-1 coordinate:", {name: round(float(scores[y == c, 0].mean()), 1) for c, name in enumerate(["ALL", "AML"])})

🔍 **What just happened?** The left panel is all the algorithm has: 72 points with no labels. Honestly, you would be
hard pressed to draw two groups in it — it looks like one broad cloud. Colouring by diagnosis (right panel) reveals that
the main axis of variation is nonetheless strongly related to the **ALL/AML difference**: the AML patients sit mostly on
the right (average PC-1 coordinate 6.7 vs −3.6 for ALL), although a few AML patients lie among the ALL patients. The two axes keep only about **13 %** and **12 %** of the total variation — most of the variation in 3,571
genes is *not* about leukaemia type (patients differ in many other ways), yet the largest single direction of variation
already carries the disease signal.

⚠️ **Hypothesis-generating, not proof.** A low-dimensional picture can suggest groups, but it can also mislead: other
choices (scaling genes, removing a batch effect) give other pictures, and an apparent cluster may reflect the lab that
processed the samples rather than biology. Golub *et al.* used a clustering method (self-organising maps) for class
discovery and then *checked* the discovered classes against the known diagnosis. Clustering is Module 12; checking
whether discovered structure is real is a recurring theme of Modules 12–14 and 23.

### 📑 Slide connection — `001_intro`, slides 52–60

- **What the slides show.** Unsupervised learning finds patterns in X without labels (clustering images or New York Times articles;
    52–54), is "much harder" than supervised learning, and is formulated as learning the density P(X) (55), which
    requires assumptions about the patterns we expect. Gene-network learning (56) and population structure from
    197,146 loci in 1,387 Europeans (57) are unsupervised. Slide 58 (Andrew Moore) summarises: classifier and regressor
    (supervised), density estimator (unsupervised); slide 59 is the course schedule; slide 60 contrasts P(Y|X) with
    P(X) and asks how to evaluate an unsupervised result.
- **What this notebook adds.** A real unsupervised analysis: a 2-D PCA map of the 72 Golub patients computed without labels, coloured only
    afterwards — the ALL/AML difference appears along the first axis, though with overlap. We also make slide 60's
    question concrete: the labels we used to check the map are exactly what an unsupervised method does not have.
- **Exam tip.** The supervised/unsupervised distinction is about the **training data** (pairs (X, Y) vs X alone), not about
    the method's name. The same Golub data support a supervised task (class prediction) and an unsupervised one (class
    discovery).

---
## 6 · What is X? What is Y? What is f?

> 📍 **Where we are.** Slides 49–57 revisit the biological examples of Section 2 and ask, for each one, the three
> questions that start every machine-learning project. Cover the answers, decide for yourself, then click to reveal.

| Problem (slide) | Your guess: X? Y? f? Supervised or unsupervised? | Answer |
|---|---|---|
| Speech recognition (49) | | <details><summary>reveal</summary>**X** = the sound recording; **Y** = the written sentence, one word per segment ("I love the Santa Cruz beach"); **f** = a speech recogniser (e.g. a hidden Markov model). Supervised: trained on recordings with transcripts.</details> |
| Gene finding (22–24, 49) | | <details><summary>reveal</summary>**X** = the DNA sequence (letters a, c, g, t); **Y** = a label for **every position** (exon, intron, intergenic); **f** = a hidden Markov model. Supervised when trained on annotated genes. Y is a whole *sequence* of labels. Modules 19–20.</details> |
| Genetic basis of disease / GWAS (30, 50) | | <details><summary>reveal</summary>**X** = a person's genotypes at many SNPs (0/1/2); **Y** = healthy vs diseased (classification) or a trait value (regression); **f** = logistic or linear regression. Supervised. Unlike gene finding, **one** label for the whole genome. Modules 06–11.</details> |
| Golub ALL vs AML (35, 51) | | <details><summary>reveal</summary>**X** = the expression of thousands of genes in one patient (a *column* of the slide's heatmap, a *row* of our table); **Y** = ALL or AML; **f** = a classifier. Supervised (class prediction); finding the classes without labels is unsupervised (class discovery). Modules 08–14.</details> |
| Gene regulatory networks (36, 56) | | <details><summary>reveal</summary>**X** = expression of many genes in hundreds of individuals; **no Y is given** — the unknown "output" is the network: which genes depend on which. Unsupervised; a probabilistic graphical model learns P(X). Modules 15–18.</details> |
| Population structure in Europe (32, 57) | | <details><summary>reveal</summary>**X** = 197,146 genotypes per person for 1,387 people; **no Y** (country labels are *not* used); output = 2 coordinates per person (and the groups they form). Unsupervised dimensionality reduction (PCA). Module 14.</details> |

✏️ **Your turn.** Invent one problem from your own biology courses — for example *"predict a protein's subcellular
location from its amino-acid sequence"* or *"group single cells from a tumour into cell types"* — and write down X, Y
(if any), f, and whether it is supervised or unsupervised, regression or classification.

---
## 7 · The course map

> 📍 **Where we are.** You have seen the two families of problems. To finish, a map of *all* the methods of 02-620 and
> the three ideas that tie them together.

**Methods × tasks.** Most methods can be placed in a table. "Generative" models describe how the data X (and Y) are
*produced*, P(X, Y) or P(X); "discriminative" models describe only P(Y | X); "non-probabilistic" methods output an
answer without a probability model (slide 68's notes: SVMs and trees "get criticised for being unable to provide
uncertainty", yet are powerful).

| Task | Non-probabilistic | Probabilistic, generative | Probabilistic, discriminative |
|---|---|---|---|
| Regression (continuous Y) | least squares as pure curve fitting | — | linear regression, ridge, lasso (06–07) |
| Classification (discrete Y) | k-nearest neighbours (08), decision trees & random forests (10), support vector machines (11) | naïve Bayes (08) | logistic regression (09), neural networks (22) |
| Clustering | k-means, hierarchical clustering (12) | Gaussian mixture models + EM (13) | — |
| Dimensionality reduction | PCA (14) | (probabilistic versions of PCA) | — |
| Networks and sequences | — | Bayesian networks (15–17), Gaussian graphical models (18), hidden Markov models (19–20) | — |

Bayesian learning and MCMC (21) and model selection (23) apply to every row.

**Three ideas that recur in every module — watch for them.**

1. **The two-step template.** *Step 1:* choose a probability model — P(Y | X; θ) for supervised problems, P(X; θ) for
   unsupervised ones — with unknown numbers θ ("theta", the **parameters**). *Step 2:* estimate θ from data, by
   **maximum likelihood (MLE)**, **maximum a posteriori (MAP)** or a full Bayesian analysis (slides 64–67; Module 05).
   Methods without a probability model (trees, k-means, SVMs) are flagged as such and paired with a probabilistic cousin.
2. **Conditional independence buys tractability.** A full probability table over many variables needs astronomically
   many numbers (3³⁰ ≈ 2 × 10¹⁴ for 30 SNPs, `003` slide 18). Assuming that some variables are independent once we know
   others shrinks this to a manageable number — the trick behind naïve Bayes, Bayesian networks, hidden Markov models
   and Gaussian graphical models (Module 02 explains "conditional independence").
3. **Hidden (latent) variables lead to the EM algorithm.** Often the most interesting quantity is never observed: the
   cluster a cell belongs to, whether a stretch of DNA is an exon. Mixture models, hidden Markov models and k-means all
   alternate between *guessing the hidden variables* and *re-estimating the parameters* (Modules 12, 13, 20, 21).

Finally, the 02-620 schedule (slide 59), with the modules that teach each week:

In [ ]:
# ── 7.1 The 02-620 schedule (slide 59) and the matching modules ──
schedule = pd.DataFrame([
    (1, "Introduction to ML", "supervised/unsupervised learning; MLE, MAP", "00, 05 (prerequisites 01–04)"),
    (2, "Regression", "linear regression, regularization", "06, 07"),
    (3, "Classification", "k-nearest neighbour; naïve Bayes; logistic regression", "08, 09"),
    (4, "Classification", "generative vs discriminative; decision tree, random forest", "09, 10"),
    (5, "Clustering", "hierarchical clustering, k-means", "12"),
    (6, "Classification", "SVM", "11"),
    (7, "Probabilistic graphical models", "Bayesian networks: models, learning, inference; Midterm I", "15, 16, 17"),
    (8, "—", "spring break", "—"),
    (9, "Latent variable models", "dynamic Bayesian networks, Gaussian graphical models", "18 (DBNs in 19–20)"),
    (10, "Latent variables; dimensionality reduction", "mixture models, EM; PCA", "13, 14"),
    (11, "Latent variable models", "hidden Markov model, forward–backward, Baum–Welch", "19, 20"),
    (12, "Latent variable models", "gene discovery from genome sequences; HMMs continued", "19, 20"),
    (13, "Bayesian learning; neural networks", "Bayesian estimation, MCMC; backpropagation", "21, 22"),
    (14, "Model selection", "bias–variance trade-off; Midterm II", "23"),
    (15, "Projects", "project presentations", "24"),
], columns=["week", "topic", "lectures", "BiologyToML modules"])
print(schedule.to_string(index=False))

🔍 **What just happened?** The table follows the order of the lectures. Note that this course reorders a few topics for
learning's sake: SVMs (week 6) come right after the other classifiers as Module 11; clustering (week 5) opens the
unsupervised part as Module 12, followed by mixture models and PCA (week 10) as Modules 13–14, *before* the graphical
models of weeks 7 and 9 (Modules 15–18). The 02-620 project — *analyse one data set with three methods* (slide 10) — is
rehearsed in the capstone, [Module 24](24_capstone_project.ipynb).

✏️ **Your turn.** Find in the table the week in which you will meet (a) the method behind the PCA map of Section 5,
(b) the method Haussler borrowed from speech recognition, (c) the first method for predicting a continuous trait from
genotypes. Which module teaches each?

## 📌 Key takeaways

- Machine learning grew where computer science meets statistics; its central idea is **learning from examples**
  (slides 12–17).
- Biology became a data science through three breakthroughs — the **genome sequence** (where are the genes? → HMMs),
  **maps of human variation** (which variants cause disease? → regression/classification; ancestry → PCA) and
  **genome-wide expression profiling** (cancer subtypes → classification/clustering; gene networks → graphical models).
- Data are tables **X** with **N rows (samples)** and **J columns (features)**; biological tables are usually **wide**
  (J ≫ N), e.g. 72 patients × 3,571 genes.
- **Supervised learning** learns f: X → Y from labelled pairs; continuous Y = **regression**, discrete Y =
  **classification**; 02-620 models it as **P(Y | X)** to express uncertainty. Evaluate on **unseen** data.
- **Unsupervised learning** has only X and models **P(X)**: it finds structure (clusters, low-dimensional maps) that
  is hypothesis-generating and harder to evaluate.
- Every module follows the same spine: **model → estimate (MLE/MAP)**, **conditional independence** to keep models
  small, **latent variables → EM**.

---
## ✅ Checkpoint

1. A notebook cell fails with `NameError: name 'golub' is not defined`. What most likely happened and how do you fix it?
   <details><summary>Answer</summary>The cell that creates `golub` (cell 3.1) has not been run in this session — perhaps the kernel was restarted. Run the cells above in order (or *Restart and run all*).</details>
2. Is predicting a patient's blood pressure from 500,000 SNPs regression or classification? Supervised or unsupervised? What is N and what is J?
   <details><summary>Answer</summary>Regression (Y is continuous), supervised (we have measured blood pressures). N = number of individuals (rows), J = 500,000 SNPs (columns).</details>
3. The Golub table has 72 rows and 3,571 columns. How many numbers is that, and why is the shape more important than the total?
   <details><summary>Answer</summary>72 × 3,571 = 257,112 numbers. The shape matters because there are far more features than samples (J ≫ N): with only 72 patients, many genes will differ between groups by chance, and some methods (e.g. ordinary least squares) break down.</details>
4. A classifier returns P(AML | x) = 0.52 for one patient and 0.97 for another. Which prediction should a doctor double-check, and why?
   <details><summary>Answer</summary>The 0.52 patient: the model is almost maximally uncertain (0.5/0.5 is a coin toss). But a high probability is not a guarantee — in Section 4 two ALL patients received P(AML|x) of 0.82 and 0.87. A toy model's probability is not a calibrated clinical risk, so in practice both would be checked by other tests.</details>
5. Why did we compute the gene ranking and the z-score scaling on the 38 training patients only?
   <details><summary>Answer</summary>Because the 34 test patients must play the role of *new* patients. Using them to choose genes or scale data leaks information about the test set into the model and makes the test accuracy over-optimistic (selection bias, Module 23).</details>
6. In the PCA map, the ALL and AML patients separate along axis 1. Does that prove that PCA 'knows' leukaemia biology?
   <details><summary>Answer</summary>No. PCA only finds the directions of largest variation; it happens that the largest one is strongly related to ALL/AML here. The map is hypothesis-generating; other sources of variation (batch, sample type) could dominate in other data.</details>
7. Gene finding and GWAS both take a DNA sequence as input. How do their outputs Y differ?
   <details><summary>Answer</summary>Gene finding outputs a label for **every position** (a sequence of labels: exon/intron/intergenic); GWAS-style disease prediction outputs **one** label (or number) for the whole genome of a person (slide 50 notes).</details>

## 🧠 Questions & discussion

1. Slide 23 suggests that a good way to make progress in biology is to recognise that your problem has been solved in another field. Which other fields' problems resemble (a) clustering single cells, (b) predicting drug response from a genome?
2. Golub's data have 72 patients and thousands of genes. What could go wrong if you tried every pair of genes and reported the pair with the best test accuracy?
3. A predicted probability is only useful if it is *calibrated*: among patients given 0.8, about 80 % should truly be AML. How could you check calibration, and how many patients would you need?
4. Unsupervised learning has no correct answers (slide 60). Suggest two ways a biologist could still decide whether the clusters found in a single-cell experiment are real.
5. UK Biobank links genotypes and health records for 500,000 people. Name one supervised and one unsupervised question you could ask of it, and one ethical issue the data raise.

## 🚀 Bonus — going further

- **Use more genes.** Replace the two genes of Section 4 by the 50 genes chosen in Section 3 (`X[:, chosen]`) and re-run the nearest-centroid classifier. Does test accuracy improve? (Keep using the training set only to choose and scale.)
- **Look at other axes.** Plot PC 2 against PC 3 (`pca_scores(X, k=3)`). Does any other structure appear? What might it be?
- **Read the original paper.** Golub *et al.* (1999), Science 286:531–537 — compare their Fig. 3B with our heatmap and their prediction-strength idea with our probabilities.
- **Related notebooks in this repository.** Generative vs discriminative classifiers on breast-cancer data: [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/); probability and uncertainty: [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/); the sister course on cryo-electron tomography: [`../../../../cryoem/tutorial/BiologyToResearch/`](../../../../cryoem/tutorial/BiologyToResearch/).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 00 — Welcome — date:
Input: Golub et al. 1999 leukaemia data (72 patients × 3,571 genes, log10; 38 train / 34 test)
Output: a 50-gene heatmap, a two-gene classifier with P(AML|x), a 2-D PCA map
Method: signal-to-noise gene ranking; nearest-centroid classification; PCA (black box)
Key equation: P(AML|x) = exp(−d²_AML/2) / [exp(−d²_ALL/2) + exp(−d²_AML/2)]
Key parameters: the two genes (GENE_A, GENE_B); the number of top genes (25 + 25)
What changed when I modified the gene pair / the ranking scale:
One failure mode: (e.g., choosing genes by test accuracy = selection bias)
Connection to 02-620 slides: 001_intro 18–37 (breakthroughs), 38–48 (supervised), 52–60 (unsupervised)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, deck `001_intro` (course design S. Kim; instructors M. Xu and M. Zhang).
- Golub T. R., Slonim D. K., Tamayo P., Huard C., Gaasenbeek M., Mesirov J. P., Coller H., Loh M. L., Downing J. R., Caligiuri M. A., Bloomfield C. D., Lander E. S. (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537.
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *Journal of the American Statistical Association* 97:77–87.
- Novembre J., Johnson T., Bryc K., Kutalik Z., Boyko A. R., Auton A., Indap A., King K. S., Bergmann S., Nelson M. R., Stephens M., Bustamante C. D. (2008) Genes mirror geography within Europe. *Nature* 456:98–101.
- International HapMap Consortium (2005) A haplotype map of the human genome. *Nature* 437:1299–1320.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 1 (introduction; supervised vs unsupervised learning).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — Chapter 1 (examples, including gene-expression data) and Chapter 2.
- Mitchell T. M. (1997) *Machine Learning*. McGraw-Hill — Chapter 1.

## ➡️ Next

[Module 01 · Python & NumPy for biologists](01_python_numpy_for_biologists.ipynb) — you have *run* code;
    now you will *write* it: DNA strings, the genetic code, genotype matrices, the Golub table in pandas, and your first
    plots.